# 📊 Aula 04 — Limpeza e Tratamento de Dados

## 🧹 Preparando os dados para a Mineração de Dados

**Disciplina:** ISW-039 — Mineração de Dados  
**Curso:** Desenvolvimento de Software Multiplataforma (DSM)  
**Ambiente:** Google Colab  
**Linguagem:** Python  
**Biblioteca principal:** Pandas

---

## 🎯 Objetivos da aula

Ao final desta aula, você deverá ser capaz de:

- Identificar problemas comuns em conjuntos de dados;
- Identificar valores ausentes;
- Identificar registros duplicados;
- Corrigir tipos de dados;
- Detectar valores inconsistentes;
- Tratar valores ausentes;
- Remover registros duplicados;
- Identificar possíveis valores extremos;
- Documentar decisões de limpeza;
- Compreender por que a qualidade dos dados influencia diretamente a Mineração de Dados.

> **Projeto didático:** continuaremos utilizando o monitoramento de motores elétricos. No projeto de avaliação, seu grupo deverá aplicar as mesmas técnicas ao seu próprio conjunto de dados.


# 🏭 1. Um problema muito comum em projetos reais

Na aula anterior trabalhamos com dados aparentemente organizados.

Na prática, isso raramente acontece.

Imagine que os sensores da indústria tenham enviado dados com problemas:

```text
temperatura = 72.5
temperatura = 71.8
temperatura = ?
temperatura = 999
temperatura = "setenta"
temperatura = 72.5
```

Antes de aplicar um algoritmo de Mineração de Dados, precisamos perguntar:

> **Podemos confiar nesses dados?**

A resposta nem sempre é sim.

Dados podem apresentar:

- valores ausentes;
- duplicidades;
- erros de digitação;
- tipos incorretos;
- valores impossíveis;
- unidades diferentes;
- registros inconsistentes;
- valores extremos.

Essa etapa é chamada de **limpeza e tratamento dos dados**.


# 🧠 2. Por que a qualidade dos dados é importante?

Imagine que queremos descobrir a temperatura média dos motores.

Se a base contém:

```text
70
72
71
73
999
```

A média será fortemente afetada pelo valor `999`.

O algoritmo não sabe que `999` provavelmente é um erro.

Ele simplesmente recebe um número.

Por isso:

> **Um algoritmo sofisticado aplicado a dados ruins pode produzir resultados ruins.**

A qualidade da análise depende da qualidade dos dados utilizados.


# 🔎 3. Problemas que vamos investigar

Nesta aula trabalharemos principalmente com:

| Problema | Exemplo |
|---|---|
| Valor ausente | temperatura = `NaN` |
| Duplicidade | mesmo registro armazenado duas vezes |
| Tipo incorreto | temperatura = `"72.5"` |
| Valor inconsistente | status = `"falh"` |
| Valor impossível | temperatura = `-300` |
| Valor extremo | temperatura = `999` |

Vamos criar uma base propositalmente problemática para aprender a identificar e tratar esses casos.


# 💻 4. Criando uma base com problemas

Observe que os erros abaixo são **intencionais**.

Eles representam situações que podem aparecer em bases reais.


In [ ]:
import pandas as pd
import numpy as np

dados_sujos = {
    "motor": [
        "M001", "M001", "M001", "M002", "M002",
        "M002", "M003", "M003", "M003", "M003",
        "M003"
    ],
    "data_hora": [
        "2026-01-01 08:00",
        "2026-01-01 09:00",
        "2026-01-01 10:00",
        "2026-01-01 08:00",
        "2026-01-01 09:00",
        "2026-01-01 09:00",
        "2026-01-01 08:00",
        "2026-01-01 09:00",
        "2026-01-01 10:00",
        "2026-01-01 11:00",
        "2026-01-01 12:00"
    ],
    "corrente": [
        12.4, 12.8, np.nan, 10.8, 11.0,
        11.0, 14.1, 14.3, 14.8, 15.1, 15.4
    ],
    "tensao": [
        380, 379, 381, 380, 381,
        381, 382, 381, 380, 379, 378
    ],
    "vibracao": [
        1.8, 2.1, 3.4, 1.2, 1.3,
        1.3, 2.4, 2.6, 2.9, 3.2, 3.8
    ],
    "temperatura": [
        62.3, 64.1, 68.7, 55.1, "56.0",
        "56.0", 67.2, 68.4, 70.1, 999, 75.3
    ],
    "rpm": [
        1750, 1748, 1742, 1752, 1750,
        1750, 1740, 1738, 1735, 1731, 1727
    ],
    "status": [
        "Normal", "Normal", "Alerta", "Normal", "Normal",
        "Normal", "Normal", "Alerta", "Alerta", "falh", "Falha"
    ]
}

df = pd.DataFrame(dados_sujos)

df

# 👀 5. Primeira inspeção

Antes de alterar qualquer coisa, devemos conhecer a base.

Uma regra importante:

> **Primeiro investigue. Depois altere.**

Vamos começar com as informações básicas.


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

Observe o resultado de `info()`.

A coluna `temperatura` provavelmente não está sendo tratada como numérica porque contém um valor armazenado como texto.

Esse é um exemplo de **problema de tipo de dado**.


# 🕳️ 6. Identificando valores ausentes

Valores ausentes são muito comuns.

No Pandas, normalmente são representados por `NaN`.

Vamos verificar quantos valores ausentes existem em cada coluna.


In [ ]:
df.isnull().sum()

Também podemos descobrir quais registros possuem valores ausentes.


In [ ]:
df[df.isnull().any(axis=1)]

## Como tratar valores ausentes?

Não existe uma única solução.

Podemos:

1. remover o registro;
2. preencher com média;
3. preencher com mediana;
4. utilizar o valor anterior ou posterior;
5. utilizar uma regra de negócio;
6. investigar a origem do problema.

A escolha depende do contexto.

No nosso exemplo, vamos utilizar a **mediana** para preencher a corrente ausente.


In [ ]:
mediana_corrente = df["corrente"].median()

df["corrente"] = df["corrente"].fillna(mediana_corrente)

print("Mediana utilizada:", mediana_corrente)
df

# 🔁 7. Identificando registros duplicados

Registros duplicados podem ocorrer quando um sistema envia o mesmo dado mais de uma vez.

Vamos verificar.


In [ ]:
df.duplicated().sum()

Vamos visualizar os registros duplicados.


In [ ]:
df[df.duplicated(keep=False)]

Se confirmarmos que o registro realmente é duplicado, podemos removê-lo.

> **Atenção:** nem toda repetição é necessariamente um erro. Precisamos analisar o significado dos dados antes de excluir registros.


In [ ]:
df = df.drop_duplicates()

df

# 🔤 8. Corrigindo tipos de dados

Vamos verificar novamente os tipos.


In [ ]:
df.dtypes

A coluna `temperatura` contém números e texto.

Podemos utilizar `pd.to_numeric()`.

O parâmetro `errors="coerce"` transforma valores que não podem ser convertidos em `NaN`.



In [ ]:
df["temperatura"] = pd.to_numeric(df["temperatura"], errors="coerce")

df.dtypes

Agora a coluna é numérica.

Vamos verificar novamente os valores ausentes.


In [ ]:
df.isnull().sum()

# 🕒 9. Convertendo datas

Datas armazenadas como texto dificultam análises temporais.

Vamos converter `data_hora`.


In [ ]:
df["data_hora"] = pd.to_datetime(df["data_hora"], errors="coerce")

df.dtypes

Agora podemos utilizar recursos próprios de datas.

Por exemplo, extrair a hora.


In [ ]:
df["hora"] = df["data_hora"].dt.hour

df[["data_hora", "hora"]]

# ⚠️ 10. Identificando valores inconsistentes

Observe a coluna `status`.

Temos:

```text
Normal
Alerta
Falha
falh
```

O valor `falh` provavelmente representa um erro de preenchimento.

Vamos verificar os valores únicos.


In [ ]:
df["status"].unique()

Podemos corrigir o valor utilizando uma regra explícita.



In [ ]:
df["status"] = df["status"].replace({
    "falh": "Falha"
})

df["status"].value_counts()

# 🚨 11. Identificando valores suspeitos

Agora precisamos investigar a temperatura.

Existe um valor `999`.

Esse valor é possível?

Não devemos simplesmente excluir qualquer valor alto.

Primeiro precisamos investigar.


In [ ]:
df.sort_values("temperatura", ascending=False)[
    ["motor", "data_hora", "temperatura", "status"]
]

O valor `999` parece suspeito.

Podemos utilizar uma regra de domínio **apenas para este exemplo didático**.

Vamos considerar temperaturas acima de 100 °C como suspeitas para investigação.

> **Importante:** esse limite não é uma especificação técnica de motores. Em um projeto real, os limites devem vir da documentação do equipamento, fabricante ou especialista do domínio.


In [ ]:
df[df["temperatura"] > 100]

# 🧹 12. Tratando o valor suspeito

Uma possibilidade é transformar o valor suspeito em ausente e depois tratá-lo.

Neste exemplo vamos utilizar `NaN`.


In [ ]:
df.loc[df["temperatura"] > 100, "temperatura"] = np.nan

df["temperatura"].isnull().sum()

Agora podemos preencher a temperatura ausente utilizando a mediana.



In [ ]:
mediana_temperatura = df["temperatura"].median()

df["temperatura"] = df["temperatura"].fillna(mediana_temperatura)

print("Mediana utilizada:", mediana_temperatura)

# 📋 13. Verificando novamente a qualidade

Depois de realizar transformações, devemos verificar a base novamente.



In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

Agora temos uma base muito mais adequada para continuar a análise.

Mas existe uma ideia importante:

> **Limpar dados não significa apagar problemas indiscriminadamente.**

Toda alteração deve possuir uma justificativa.


# 📝 14. Exercícios

## Exercício 1 — Diagnóstico

Crie um diagnóstico inicial da base.

Mostre:

- número de linhas;
- número de colunas;
- tipos de dados;
- quantidade de valores ausentes;
- quantidade de duplicidades.



In [ ]:
# Sua resposta



## Exercício 2 — Valores ausentes

Identifique todas as colunas que possuem valores ausentes.

Depois explique qual estratégia você utilizaria para tratar cada uma delas.


In [ ]:
# Sua resposta



## Exercício 3 — Duplicidade

Verifique se ainda existem registros duplicados.

Quantos existem?


In [ ]:
# Sua resposta



## Exercício 4 — Tipos

Verifique os tipos das colunas.

Qual coluna precisou de tratamento?

Por que o tipo correto é importante?


In [ ]:
# Sua resposta



## Exercício 5 — Status

Liste os valores únicos da coluna `status`.

Existe alguma inconsistência?

Corrija-a e mostre o resultado.


In [ ]:
# Sua resposta



## Exercício 6 — Datas

Converta `data_hora` para o tipo datetime e crie uma coluna chamada `dia`.

Depois mostre:

```text
data_hora
dia
```


In [ ]:
# Sua resposta



## Exercício 7 — Investigação de valores extremos

Calcule:

- temperatura mínima;
- temperatura máxima;
- corrente mínima;
- corrente máxima;
- vibração mínima;
- vibração máxima.

Depois indique quais valores merecem investigação.


In [ ]:
# Sua resposta



## Exercício 8 — Regra de negócio

Crie uma regra para identificar registros que merecem atenção.

Por exemplo:

```text
temperatura > 70
OU
vibracao > 3
```

Crie uma coluna chamada `investigar` com:

- `Sim`
- `Não`



In [ ]:
# Sua resposta



## Exercício 9 — Antes e depois

Crie uma tabela contendo:

- motor;
- temperatura;
- vibração;
- corrente;
- status;
- investigar.

Ordene pela maior temperatura.

Analise os cinco primeiros registros.


In [ ]:
# Sua resposta



## Exercício 10 — Documentação

Imagine que você precise explicar para outra pessoa todas as alterações realizadas na base.

Crie uma tabela ou lista contendo:

| Problema | Tratamento realizado | Justificativa |
|---|---|---|
| Valor ausente | ... | ... |
| Duplicidade | ... | ... |
| Tipo incorreto | ... | ... |
| Status inconsistente | ... | ... |
| Valor suspeito | ... | ... |



In [ ]:
# Sua resposta



# 🔎 15. Desafio — Auditoria da base

Agora imagine que você recebeu uma base de sensores sem nenhuma documentação.

Seu trabalho é criar uma pequena **rotina de auditoria**.

O programa deverá informar:

1. Quantidade de registros;
2. Quantidade de colunas;
3. Valores ausentes por coluna;
4. Quantidade de duplicados;
5. Tipos das colunas;
6. Valores únicos das variáveis categóricas;
7. Mínimo e máximo das variáveis numéricas.

### Objetivo

Criar uma função chamada:

```python
auditar_dados(df)
```

que execute essas verificações.



In [ ]:
def auditar_dados(df):
    # Desenvolva sua solução aqui
    pass

# Teste:
# auditar_dados(df)


# 🚀 16. Aplicação no seu projeto

Agora leve o conceito para o projeto que seu grupo está começando a definir.

Pense na futura base de dados do seu projeto.

### Faça uma lista de possíveis problemas

Para cada problema, responda:

1. Que tipo de erro pode aparecer?
2. Como você poderia identificá-lo?
3. Como poderia tratá-lo?
4. Como justificaria essa decisão?

### Exemplo

**Projeto:** previsão de vendas

**Problema:** preço negativo

**Identificação:** procurar valores menores que zero

**Tratamento:** investigar/remover/corrigir conforme a origem

**Justificativa:** preço negativo não representa uma venda válida.

> Essa atividade será importante mais adiante, quando seu grupo começar a trabalhar com os próprios dados.


In [ ]:
# Registre aqui os possíveis problemas encontrados no seu projeto.



# 📌 17. Checklist da Aula

Antes de finalizar o notebook:

- [ ] Sei identificar valores ausentes;
- [ ] Sei verificar duplicidades;
- [ ] Sei verificar tipos de dados;
- [ ] Sei converter texto para número;
- [ ] Sei converter texto para data;
- [ ] Sei identificar valores inconsistentes;
- [ ] Sei investigar valores suspeitos;
- [ ] Sei utilizar `fillna()`;
- [ ] Sei utilizar `drop_duplicates()`;
- [ ] Sei utilizar `replace()`;
- [ ] Sei documentar decisões de limpeza;
- [ ] Entendo que uma alteração nos dados precisa ser justificada.

---

# 🎯 Conclusão

Na Aula 3 aprendemos a **manipular dados**.

Nesta aula aprendemos a **avaliar e melhorar a qualidade desses dados**.

O fluxo começa a ficar mais claro:

```text
Problema
   ↓
Dados
   ↓
Entendimento
   ↓
Limpeza
   ↓
Dados preparados
   ↓
Mineração de Dados
```

Na próxima aula vamos avançar para **ETL — Extract, Transform and Load**, entendendo como os dados podem ser extraídos de diferentes fontes, transformados e preparados para análise.
